# KATS — Experiment 5: Ablation Study

KATS Framework — Kinetic Attack Triage System


In [ ]:
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV
import lightgbm as lgb

ablation_results = []

def run_ablation(name, estimator, X, y, dep_features=None):
    """Run 5-fold CV and return metrics dict."""
    X_run = X.copy()
    if dep_features is not None:
        X_run = X_run.drop(columns=dep_features, errors='ignore')

    cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y), dtype=int)
    for tr, te in cv5.split(X_run, y):
        estimator.fit(X_run.iloc[tr], y.iloc[tr])
        oof[te] = estimator.predict(X_run.iloc[te])

    return {
        'Ablation':       name,
        'Recall_High':    round(recall_score(y, oof, labels=[2], average='macro', zero_division=0), 4),
        'Macro_F1':       round(f1_score(y, oof, average='macro', zero_division=0), 4),
        'Kappa':          round(cohen_kappa_score(y, oof), 4),
        'Precision_High': round(precision_score(y, oof, labels=[2], average='macro', zero_division=0), 4),
    }

# ── T5.1 Full KATS-Ensemble (baseline for comparison) ────────────────────
print("Running T5.0: Full KATS-Ensemble...")
rf_a    = RandomForestClassifier(n_estimators=200, max_depth=15, class_weight={0:1,1:1,2:5}, random_state=42, n_jobs=-1)
lgbm_a  = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, class_weight={0:1,1:1,2:5}, random_state=42, n_jobs=-1, verbose=-1)
nb_a    = CalibratedClassifierCV(GaussianNB(), method='isotonic', cv=3)
meta_a  = LogisticRegression(class_weight={0:1,1:1,2:5}, max_iter=1000, random_state=42)
full    = Pipeline([('sc', StandardScaler()), ('m', StackingClassifier(
    estimators=[('rf',rf_a),('lgbm',lgbm_a),('nb',nb_a)],
    final_estimator=meta_a, passthrough=True, cv=5, n_jobs=-1))])
ablation_results.append(run_ablation('T5.0 Full KATS-Ensemble', full, X_syn, y_syn))
print(f"  Recall_High: {ablation_results[-1]['Recall_High']}")

# ── T5.1 No Asymmetric Loss (uniform class weights) ───────────────────────
print("Running T5.1: No Asymmetric Loss...")
rf_1   = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
lgbm_1 = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, random_state=42, n_jobs=-1, verbose=-1)
nb_1   = CalibratedClassifierCV(GaussianNB(), method='isotonic', cv=3)
meta_1 = LogisticRegression(max_iter=1000, random_state=42)  # NO class weight
t51    = Pipeline([('sc', StandardScaler()), ('m', StackingClassifier(
    estimators=[('rf',rf_1),('lgbm',lgbm_1),('nb',nb_1)],
    final_estimator=meta_1, passthrough=True, cv=5, n_jobs=-1))])
ablation_results.append(run_ablation('T5.1 No Asymmetric Loss (α=1)', t51, X_syn, y_syn))
print(f"  Recall_High: {ablation_results[-1]['Recall_High']}")

# ── T5.2 No Calibrated NB (RF + LightGBM only) ───────────────────────────
print("Running T5.2: No Calibrated NB...")
rf_2   = RandomForestClassifier(n_estimators=200, max_depth=15, class_weight={0:1,1:1,2:5}, random_state=42, n_jobs=-1)
lgbm_2 = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, class_weight={0:1,1:1,2:5}, random_state=42, n_jobs=-1, verbose=-1)
meta_2 = LogisticRegression(class_weight={0:1,1:1,2:5}, max_iter=1000, random_state=42)
t52    = Pipeline([('sc', StandardScaler()), ('m', StackingClassifier(
    estimators=[('rf',rf_2),('lgbm',lgbm_2)],   # ← NB removed
    final_estimator=meta_2, passthrough=True, cv=5, n_jobs=-1))])
ablation_results.append(run_ablation('T5.2 No Calibrated NB', t52, X_syn, y_syn))
print(f"  Recall_High: {ablation_results[-1]['Recall_High']}")

# ── T5.3 Single Model — RF only ───────────────────────────────────────────
print("Running T5.3: Single RF only...")
t53 = Pipeline([('sc', StandardScaler()),
                ('m',  RandomForestClassifier(n_estimators=300, max_depth=15,
                                               class_weight={0:1,1:1,2:5},
                                               random_state=42, n_jobs=-1))])
ablation_results.append(run_ablation('T5.3 Single Model (RF only)', t53, X_syn, y_syn))
print(f"  Recall_High: {ablation_results[-1]['Recall_High']}")

# ── T5.5 No Dependency Features ──────────────────────────────────────────
print("Running T5.5: No Dependency Features...")
rf_5   = RandomForestClassifier(n_estimators=200, max_depth=15, class_weight={0:1,1:1,2:5}, random_state=42, n_jobs=-1)
lgbm_5 = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, class_weight={0:1,1:1,2:5}, random_state=42, n_jobs=-1, verbose=-1)
nb_5   = CalibratedClassifierCV(GaussianNB(), method='isotonic', cv=3)
meta_5 = LogisticRegression(class_weight={0:1,1:1,2:5}, max_iter=1000, random_state=42)
stack_5 = StackingClassifier(estimators=[('rf',rf_5),('lgbm',lgbm_5),('nb',nb_5)],
                               final_estimator=meta_5, passthrough=True, cv=5, n_jobs=-1)
# Drop dependency features BEFORE scaler
dep_cols = ['dependency_count', 'downstream_critical']
X_no_dep = X_syn.drop(columns=dep_cols)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_5 = np.zeros(len(y_syn), dtype=int)
for tr, te in cv5.split(X_no_dep, y_syn):
    pipe_5 = Pipeline([('sc', StandardScaler()), ('m', StackingClassifier(
        estimators=[('rf', RandomForestClassifier(n_estimators=200, max_depth=15, class_weight={0:1,1:1,2:5}, random_state=42, n_jobs=-1)),
                    ('lgbm', lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, class_weight={0:1,1:1,2:5}, random_state=42, n_jobs=-1, verbose=-1)),
                    ('nb', CalibratedClassifierCV(GaussianNB(), method='isotonic', cv=3))],
        final_estimator=LogisticRegression(class_weight={0:1,1:1,2:5}, max_iter=1000, random_state=42),
        passthrough=True, cv=3, n_jobs=-1))])
    pipe_5.fit(X_no_dep.iloc[tr], y_syn.iloc[tr])
    oof_5[te] = pipe_5.predict(X_no_dep.iloc[te])

ablation_results.append({
    'Ablation':       'T5.5 No Dependency Features',
    'Recall_High':    round(recall_score(y_syn, oof_5, labels=[2], average='macro', zero_division=0), 4),
    'Macro_F1':       round(f1_score(y_syn, oof_5, average='macro', zero_division=0), 4),
    'Kappa':          round(cohen_kappa_score(y_syn, oof_5), 4),
    'Precision_High': round(precision_score(y_syn, oof_5, labels=[2], average='macro', zero_division=0), 4),
})
print(f"  Recall_High: {ablation_results[-1]['Recall_High']}")

df_ablation = pd.DataFrame(ablation_results)
df_ablation.to_csv('/kaggle/working/experiment5_ablation.csv', index=False)

print("\n" + "="*72)
print("EXPERIMENT 5 — ABLATION STUDY (Sorted by Recall_High)")
print("="*72)
print(df_ablation.sort_values('Recall_High', ascending=False).to_string(index=False))

# Compute drops vs full model
full_recall = df_ablation[df_ablation['Ablation'].str.contains('Full')]['Recall_High'].values[0]
print(f"\n📊 Recall_High Drops vs Full Model:")
for _, row in df_ablation.iterrows():
    if 'Full' not in row['Ablation']:
        drop = full_recall - row['Recall_High']
        print(f"  {row['Ablation']:<40} drop = {drop:+.4f} ({drop*100:.2f} pp)")